# Pittsburgh LVT Policy Example

Pittsburgh is one of the most historically significant cities for Land Value Taxation in the United States. From approximately 1913 to 2001, Pittsburgh operated a **graded tax** (split-rate property tax) that taxed land at roughly twice the rate applied to buildings—one of the longest-running LVT experiments in American history. The program was abolished in 2001 primarily due to assessment inequities and political opposition, not because of economic failure.

Pennsylvania is the only state that explicitly authorizes municipalities to adopt split-rate property taxation, and many smaller PA cities still use it today (Harrisburg, Allentown, Scranton, etc.).

This notebook models an LVT shift for **Pittsburgh city taxes only** (analogous to the `st_paul_v2.ipynb` approach), holding county and school district millages constant. Pittsburgh's three taxing bodies in 2025 are:

| Taxing Body | 2025 Millage (mills) |
|---|---|
| Allegheny County | 6.43 |
| City of Pittsburgh | 8.06 |
| Pittsburgh School District | 10.25 |
| **Combined** | **24.74** |

**Data source**: Allegheny County property assessment data from the Western Pennsylvania Regional Data Center (WPRDC). Assessment values are from the 2012 base-year county-wide reassessment (the last full reassessment). The Common Level Ratio (CLR) for 2025 is 52.7%, meaning current market values are roughly 1.90× the assessed values in this dataset.

**Assessment system**: Allegheny County maintains separate assessed values for county vs. local (city/school) taxes. The `LOCALTOTAL` field is the taxable base for Pittsburgh city and school district taxes. The homestead exclusion reduces `LOCALBUILDING` (not `LOCALLAND`) by $15,000 for eligible owner-occupied primary residences.


In [1]:
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv
from shapely.geometry import Polygon, MultiPolygon

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "lvt_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt_utils import (
    calculate_category_tax_summary,
    calculate_current_tax,
    model_split_rate_tax,
    print_category_tax_summary,
)
from policy_analysis import (
    analyze_parking_lots,
    analyze_vacant_land,
    print_parking_analysis_summary,
    print_vacant_land_summary,
)
from viz import calculate_block_group_summary

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

In [ ]:
# ---- Configuration ----

# 2025 Allegheny County millage rates (mills = per $1,000 assessed value)
COUNTY_MILLAGE = 6.43
CITY_MILLAGE = 8.06       # Pittsburgh city levy (2025)
SCHOOL_MILLAGE = 10.25    # Pittsburgh school district (2025)
COMBINED_MILLAGE = COUNTY_MILLAGE + CITY_MILLAGE + SCHOOL_MILLAGE  # 24.74 total

# Common Level Ratio 2025 (Pennsylvania STEB): market_value ≈ assessed_value / CLR
CLR_2025 = 0.527

# Pittsburgh MUNICODE range: wards 1-32 → codes 101-132 in the Allegheny County system
PITTSBURGH_MUNICODES = list(range(101, 133))
PITTSBURGH_MUNICODE_STR = ",".join(str(m) for m in PITTSBURGH_MUNICODES)

# Allegheny County FIPS for Census
ALLEGHENY_FIPS = "42003"

# Data directories
data_dir = REPO_ROOT / "examples" / "data" / "pittsburgh"
data_dir.mkdir(parents=True, exist_ok=True)

# Data scrape flag: 1 = fetch fresh data, 0 = load from cache
data_scrape = 1

print(f"Data directory: {data_dir}")
print(f"Combined millage: {COMBINED_MILLAGE} mills")
print(f"City millage (modeled): {CITY_MILLAGE} mills")

Data directory: /Users/gregmiller/Documents/CLE/cle/LVTShift/examples/data/pittsburgh
Combined millage: 24.740000000000002 mills
City millage (modeled): 8.06 mills


## Step 1: Load Parcel Assessment Data

Allegheny County property assessment data is published by the Western Pennsylvania Regional Data Center (WPRDC). The master file contains all ~583,000 county parcels with:
- Parcel identifiers: `PARID` (16-character, join key), `MUNICODE`, `MUNIDESC`
- Fair Market Values (2012 base year): `FAIRMARKETLAND`, `FAIRMARKETBUILDING`, `FAIRMARKETTOTAL`
- County assessed values (after $18K homestead reduction): `COUNTYLAND`, `COUNTYBUILDING`, `COUNTYTOTAL`
- Local assessed values (after $15K homestead reduction): `LOCALLAND`, `LOCALBUILDING`, `LOCALTOTAL`
- Tax status: `TAXCODE` (E=Exempt, T=Taxable, P=PURTA utility)
- Property class: `CLASS` (R=Residential, C=Commercial, I=Industrial, V=Vacant, G=Government)
- Use description: `USEDESC` (granular land use text)
- Homestead flag: `HOMESTEADFLAG` ('HOM' if homestead exclusion applies)

Pittsburgh parcels are filtered to `MUNICODE` values 101–132 (Pittsburgh's 32 wards).

**WPRDC master file**: https://data.wprdc.org/dataset/property-assessments

In [3]:
# WPRDC master CSV URL (full Allegheny County, ~580K rows)
WPRDC_CSV_URL = (
    "https://data.wprdc.org/dataset/2b3df818-601e-4f06-b150-643557229491"
    "/resource/9a1c60bd-f9f7-4aba-aeb7-af8c3aaa44e5"
    "/download/allegheny_county_master_file.csv"
)

attrs_cache = data_dir / "pittsburgh_attrs.parquet"


def fetch_wprdc_pittsburgh():
    """Download WPRDC master CSV and filter to Pittsburgh parcels."""
    print("Downloading WPRDC master file (may take several minutes)...")
    # Stream to avoid loading the full 500MB+ file into memory all at once
    chunks = []
    for chunk in pd.read_csv(
        WPRDC_CSV_URL,
        chunksize=50_000,
        dtype={"PARID": str, "MUNICODE": "Int64"},
        low_memory=False,
    ):
        pgh = chunk[chunk["MUNICODE"].isin(PITTSBURGH_MUNICODES)]
        if len(pgh) > 0:
            chunks.append(pgh)
        print(f"  processed chunk, Pittsburgh rows so far: {sum(len(c) for c in chunks):,}")
    df = pd.concat(chunks, ignore_index=True)
    print(f"Pittsburgh parcel count from WPRDC: {len(df):,}")
    # Fix mixed-type object columns for parquet compatibility
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).replace({"nan": None, "<NA>": None})
    return df


if data_scrape == 1:
    parcel_attrs = fetch_wprdc_pittsburgh()
    parcel_attrs.to_parquet(attrs_cache, index=False)
    print(f"Saved to {attrs_cache}")
else:
    parcel_attrs = pd.read_parquet(attrs_cache)
    print(f"Loaded {len(parcel_attrs):,} Pittsburgh parcels from cache")

display(parcel_attrs.head(3))
print("\nColumns:", parcel_attrs.columns.tolist())

Loaded 142,470 Pittsburgh parcels from cache


,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,SCHOOLCODE,SCHOOLDESC,LEGAL1,LEGAL2,LEGAL3,NEIGHCODE,NEIGHDESC,TAXCODE,TAXDESC,TAXSUBCODE,TAXSUBCODE_DESC,OWNERCODE,OWNERDESC,CLASS,CLASSDESC,USECODE,USEDESC,LOTAREA,HOMESTEADFLAG,FARMSTEADFLAG,CLEANGREEN,ABATEMENTFLAG,RECORDDATE,SALEDATE,SALEPRICE,SALECODE,SALEDESC,DEEDBOOK,DEEDPAGE,PREVSALEDATE,PREVSALEPRICE,PREVSALEDATE2,PREVSALEPRICE2,CHANGENOTICEADDRESS1,CHANGENOTICEADDRESS2,CHANGENOTICEADDRESS3,CHANGENOTICEADDRESS4,COUNTYBUILDING,COUNTYLAND,COUNTYTOTAL,COUNTYEXEMPTBLDG,LOCALBUILDING,LOCALLAND,LOCALTOTAL,FAIRMARKETBUILDING,FAIRMARKETLAND,FAIRMARKETTOTAL,STYLE,STYLEDESC,STORIES,YEARBLT,EXTERIORFINISH,EXTFINISH_DESC,ROOF,ROOFDESC,BASEMENT,BASEMENTDESC,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE
0,0001G00162000000,601.0,,COMMONWEALTH PL,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 180 X 159.8 X 137.07,None,None,51C01,PITTSBURGH URBAN,E,10 - Exempt,None,None,20,CORPORATION,G,GOVERNMENT,610,STATE GOVERNMENT,39960,None,None,None,None,None,12-30-1991,1.0,3,LOVE&AFFECTION,8630,499,None,NaN,None,NaN,300 LIBERTY AVE,,PITTSBURGH PA,15222.0,0,2497500,2497500,0,0,2497500,2497500,0,2497500,2497500,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,None,None,NaN,None,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,2026,01-MAR-26
1,0001G00193000000,100.0,,STANWIX ST,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 51.67X80,None,None,51C01,PITTSBURGH URBAN,T,20 - Taxable,None,None,20,CORPORATION,C,COMMERCIAL,447,OFFICE - 1-2 STORIES,8320,None,None,None,None,10-30-2025,10-27-2025,1400000.0,H,MULTI-PARCEL SA,20226,72,12-28-2012,1.0,02-06-1998,493451.0,1426 ORCHARDVIEW DR,,PITTSBURGH PA,15220.0,224500,499200,723700,0,224500,499200,723700,224500,499200,723700,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,None,None,NaN,None,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,2026,01-MAR-26
2,0001G00077000001,202.0,,STANWIX ST,PITTSBURGH,PA,,15222.0,101,1st Ward - PITTSBURGH,47,Pittsburgh,ST MARYS CHURCH SUBDIVISION PLAN,LOT 1=80.15X120.40X80.15X120.40,None,51C01,PITTSBURGH URBAN,T,20 - Taxable,None,None,20,CORPORATION,C,COMMERCIAL,685,"CHURCHES, PUBLIC WORSHIP",9649,None,None,None,Y,None,03-24-1920,0.0,3,LOVE&AFFECTION,1998,540,None,NaN,None,NaN,202 STANWIX ST,,PITTSBURGH PA,15222.0,102560,34830,137390,1362640,102560,34830,137390,1465200,497600,1962800,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,None,None,NaN,None,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,2026,01-MAR-26



Columns: ['PARID', 'PROPERTYHOUSENUM', 'PROPERTYFRACTION', 'PROPERTYADDRESS', 'PROPERTYCITY', 'PROPERTYSTATE', 'PROPERTYUNIT', 'PROPERTYZIP', 'MUNICODE', 'MUNIDESC', 'SCHOOLCODE', 'SCHOOLDESC', 'LEGAL1', 'LEGAL2', 'LEGAL3', 'NEIGHCODE', 'NEIGHDESC', 'TAXCODE', 'TAXDESC', 'TAXSUBCODE', 'TAXSUBCODE_DESC', 'OWNERCODE', 'OWNERDESC', 'CLASS', 'CLASSDESC', 'USECODE', 'USEDESC', 'LOTAREA', 'HOMESTEADFLAG', 'FARMSTEADFLAG', 'CLEANGREEN', 'ABATEMENTFLAG', 'RECORDDATE', 'SALEDATE', 'SALEPRICE', 'SALECODE', 'SALEDESC', 'DEEDBOOK', 'DEEDPAGE', 'PREVSALEDATE', 'PREVSALEPRICE', 'PREVSALEDATE2', 'PREVSALEPRICE2', 'CHANGENOTICEADDRESS1', 'CHANGENOTICEADDRESS2', 'CHANGENOTICEADDRESS3', 'CHANGENOTICEADDRESS4', 'COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL', 'COUNTYEXEMPTBLDG', 'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL', 'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL', 'STYLE', 'STYLEDESC', 'STORIES', 'YEARBLT', 'EXTERIORFINISH', 'EXTFINISH_DESC', 'ROOF', 'ROOFDESC', 'BASEMENT', 'BASEMENTDESC',

In [4]:
# Allegheny County OPENDATA parcels MapServer (geometry only: PIN, MUNICODE, MAPBLOCKLOT)
GIS_BASE_URL = "https://gisdata.alleghenycounty.us/arcgis/rest/services/OPENDATA/Parcels/MapServer"
GIS_LAYER_ID = 0
geometry_cache = data_dir / "pittsburgh_geometry.parquet"


def fetch_pittsburgh_geometry():
    """Fetch parcel polygon geometries for Pittsburgh from Allegheny County GIS."""
    query_url = f"{GIS_BASE_URL}/{GIS_LAYER_ID}/query"
    where = f"MUNICODE IN ({PITTSBURGH_MUNICODE_STR})"

    # Count records
    count_resp = requests.get(
        query_url,
        params={"f": "json", "where": where, "returnCountOnly": "true"},
        timeout=60,
    )
    count_resp.raise_for_status()
    total = count_resp.json().get("count", 0)
    print(f"Pittsburgh parcel geometries to fetch: {total:,}")

    all_features = []
    offset = 0
    chunk_size = 1000

    while offset < total:
        params = {
            "f": "json",
            "where": where,
            "outFields": "PIN,MUNICODE,MAPBLOCKLOT",
            "returnGeometry": "true",
            "outSR": "4326",
            "geometryPrecision": 6,
            "resultOffset": offset,
            "resultRecordCount": chunk_size,
        }
        resp = requests.get(query_url, params=params, timeout=120)
        resp.raise_for_status()
        data = resp.json()

        features = data.get("features", [])
        if not features:
            break

        for feat in features:
            attrs = feat["attributes"].copy()
            geom_data = feat.get("geometry", {})
            geometry = None
            if geom_data and "rings" in geom_data and geom_data["rings"]:
                rings = geom_data["rings"]
                if len(rings) == 1:
                    geometry = Polygon(rings[0])
                else:
                    # Multi-ring: first ring = exterior, remaining = holes
                    try:
                        exterior = rings[0]
                        holes = rings[1:]
                        geometry = Polygon(exterior, holes)
                    except Exception:
                        geometry = Polygon(rings[0])
            attrs["geometry"] = geometry
            all_features.append(attrs)

        print(f"  fetched {offset} to {offset + len(features)} of {total}")
        if len(features) < chunk_size:
            break
        offset += len(features)
        time.sleep(0.2)

    gdf = gpd.GeoDataFrame(all_features, geometry="geometry", crs="EPSG:4326")
    valid = gdf["geometry"].notna().sum()
    print(f"Geometries with valid polygons: {valid:,} of {len(gdf):,}")
    return gdf


if data_scrape == 1:
    parcel_geometry = fetch_pittsburgh_geometry()
    parcel_geometry.to_parquet(geometry_cache, index=False)
    print(f"Saved geometry to {geometry_cache}")
else:
    parcel_geometry = gpd.read_parquet(geometry_cache)
    print(f"Loaded {len(parcel_geometry):,} parcel geometries from cache")

display(parcel_geometry.head(3))

FileNotFoundError: [Errno 2] Failed to open local file '/Users/gregmiller/Documents/CLE/cle/LVTShift/examples/data/pittsburgh/pittsburgh_geometry.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
# Normalize PARID and PIN for joining.
# WPRDC PARID may have dashes or spaces; GIS PIN is typically without dashes.
# Normalize both by stripping all non-alphanumeric characters and uppercasing.
parcel_attrs["PARID_norm"] = parcel_attrs["PARID"].astype(str).str.replace(r"[^A-Z0-9]", "", regex=True).str.upper()
parcel_geometry["PIN_norm"] = parcel_geometry["PIN"].astype(str).str.replace(r"[^A-Z0-9]", "", regex=True).str.upper()

# Check uniqueness before joining
print(f"Unique PARID_norm in attrs: {parcel_attrs['PARID_norm'].nunique():,} of {len(parcel_attrs):,}")
print(f"Unique PIN_norm in geometry: {parcel_geometry['PIN_norm'].nunique():,} of {len(parcel_geometry):,}")

# Merge: keep all attribute rows, bring in geometry where available
gdf = parcel_attrs.merge(
    parcel_geometry[["PIN_norm", "geometry"]],
    left_on="PARID_norm",
    right_on="PIN_norm",
    how="left",
)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
gdf = gdf.drop_duplicates(subset=["PARID"]).reset_index(drop=True)

print(f"\nMerged GDF: {len(gdf):,} rows")
print(f"Rows with geometry: {gdf['geometry'].notna().sum():,}")
print(f"Rows without geometry: {gdf['geometry'].isna().sum():,}")
display(gdf[["PARID", "MUNICODE", "MUNIDESC", "CLASS", "USEDESC", "LOCALTOTAL", "geometry"]].head(5))

## Step 2: Data Exploration

Explore the Allegheny County field structure to understand:
- How properties are classified (`CLASS`, `USEDESC`)
- Distribution of assessed values
- Exempt vs taxable parcel counts

In [ ]:
numeric_cols = [
    "FAIRMARKETLAND", "FAIRMARKETBUILDING", "FAIRMARKETTOTAL",
    "COUNTYLAND", "COUNTYBUILDING", "COUNTYTOTAL",
    "LOCALLAND", "LOCALBUILDING", "LOCALTOTAL",
]
for col in numeric_cols:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce").fillna(0)

print("TAX CODE distribution:")
print(gdf["TAXCODE"].value_counts(dropna=False))
print("\nTAXDESC distribution:")
print(gdf["TAXDESC"].value_counts(dropna=False).head(10))

print("\nCLASS distribution:")
print(gdf["CLASS"].value_counts(dropna=False))

print("\nTop 30 USEDESC values:")
display(gdf["USEDESC"].value_counts(dropna=False).head(30))

In [ ]:
# Summarize assessment values for taxable parcels only
taxable = gdf[gdf["TAXCODE"] == "T"]
print(f"Taxable parcel count: {len(taxable):,}")
print(f"\nAggregate taxable values (TAXCODE='T'):")
print(f"  FAIRMARKETTOTAL sum:  ${taxable['FAIRMARKETTOTAL'].sum():>15,.0f}")
print(f"  COUNTYTOTAL sum:      ${taxable['COUNTYTOTAL'].sum():>15,.0f}")
print(f"  LOCALTOTAL sum:       ${taxable['LOCALTOTAL'].sum():>15,.0f}")
print(f"\n  LOCALLAND sum:        ${taxable['LOCALLAND'].sum():>15,.0f}")
print(f"  LOCALBUILDING sum:    ${taxable['LOCALBUILDING'].sum():>15,.0f}")
print(f"  (LOCALLAND+LOCALBUILDING):  ${(taxable['LOCALLAND']+taxable['LOCALBUILDING']).sum():>15,.0f}")

# Compute improvement ratio
total_land = taxable["LOCALLAND"].sum()
total_building = taxable["LOCALBUILDING"].sum()
total_local = taxable["LOCALTOTAL"].sum()
ir = total_building / total_local if total_local > 0 else 0
print(f"\nAggregate improvement ratio (building/total): {ir:.3f}")
print(f"Aggregate land ratio (land/total):            {1 - ir:.3f}")

# Homestead exemption summary
if "HOMESTEADFLAG" in gdf.columns:
    hs_count = (gdf["HOMESTEADFLAG"] == "HOM").sum()
    print(f"\nHomestead-eligible parcels: {hs_count:,}")

## Step 3: Property Classification and Exemption Handling

### Allegheny County CLASS codes
- `R` — Residential (1–4 dwelling units, single-family and small multi-family)
- `C` — Commercial (5+ unit apartment buildings are coded commercial here)
- `I` — Industrial
- `V` — Vacant land
- `A` / `F` — Agricultural / Farmstead
- `G` — Government-owned
- `U` — Utilities
- `O` — Other

**Key distinction**: Allegheny County codes all residential uses with 5+ units under `CLASS='C'`. These must be identified via `USEDESC` to assign them to `Large Multi-Family (5+ units)` rather than `Commercial`.

**Exemptions**: Full exemptions are `TAXCODE='E'`. Partial exemptions (homestead) are already baked into `LOCALBUILDING` and `LOCALLAND`. `TAXCODE='P'` (PURTA utility tax) parcels are treated as taxable for modeling purposes since they pay a state-administered levy, but they are flagged separately.

**Vacant land override**: Any taxable parcel with `FAIRMARKETBUILDING=0` and not already classified as Vacant is reclassified to Vacant Land.

In [ ]:
# Property category mapping for Allegheny County
# Based on CLASS field + USEDESC text patterns.
# Run Step 2 first to examine actual USEDESC values in the data.

USEDESC_CATEGORY_MAP = {
    # Single Family
    "SINGLE FAMILY": "Single Family Residential",
    "ONE FAMILY": "Single Family Residential",
    "1 FAMILY": "Single Family Residential",
    "ROWHOUSE": "Single Family Residential",
    "ROW HOUSE": "Single Family Residential",
    "TOWNHOUSE": "Single Family Residential",
    "TOWN HOUSE": "Single Family Residential",
    "CONDO": "Single Family Residential",
    "CONDOMINIUM": "Single Family Residential",
    # Small Multi-Family (2-4 units)
    "TWO FAMILY": "Small Multi-Family (2-4 units)",
    "2 FAMILY": "Small Multi-Family (2-4 units)",
    "THREE FAMILY": "Small Multi-Family (2-4 units)",
    "3 FAMILY": "Small Multi-Family (2-4 units)",
    "FOUR FAMILY": "Small Multi-Family (2-4 units)",
    "4 FAMILY": "Small Multi-Family (2-4 units)",
    "DUPLEX": "Small Multi-Family (2-4 units)",
    "TRIPLEX": "Small Multi-Family (2-4 units)",
    # Large Multi-Family (5+ units) — coded as CLASS='C' in Allegheny County
    # APART catches: APART: 5-19 UNITS, APART:20-39 UNITS, APART:40+ UNITS
    "APART": "Large Multi-Family (5+ units)",
    "APARTMENT": "Large Multi-Family (5+ units)",
    "APTS": "Large Multi-Family (5+ units)",
    "APT": "Large Multi-Family (5+ units)",
    "MULTI FAMILY": "Large Multi-Family (5+ units)",
    "MULTIFAMILY": "Large Multi-Family (5+ units)",
    # Parking
    "PARKING LOT": "Transportation - Parking",
    "PARKING GARAGE": "Transportation - Parking",
    "GARAGE": "Transportation - Parking",
    "PARKING": "Transportation - Parking",
    # Vacant
    "VACANT LAND": "Vacant Land",
    "VACANT": "Vacant Land",
    "VAC LAND": "Vacant Land",
    "UNIMPROVED": "Vacant Land",
}


def assign_category(row):
    """Assign PROPERTY_CATEGORY from CLASS + USEDESC."""
    tax_code = str(row.get("TAXCODE", "")).strip().upper()
    cls = str(row.get("CLASS", "")).strip().upper()
    usedesc = str(row.get("USEDESC", "")).strip().upper()

    # Fully exempt
    if tax_code == "E":
        return "Exempt"

    # Check USEDESC keyword mapping first (more specific)
    for keyword, category in USEDESC_CATEGORY_MAP.items():
        if keyword in usedesc:
            return category

    # Fall back to CLASS
    class_map = {
        "R": "Single Family Residential",
        "C": "Commercial",
        "I": "Industrial",
        "V": "Vacant Land",
        "A": "Agricultural",
        "F": "Agricultural",
        "G": "Other",
        "U": "Other",
        "O": "Other",
    }
    return class_map.get(cls, "Other")


gdf["PROPERTY_CATEGORY"] = gdf.apply(assign_category, axis=1)

# Override: parcels with zero improvement value → Vacant Land (except Exempt)
vacant_override_mask = (
    (gdf["FAIRMARKETBUILDING"] == 0)
    & (gdf["PROPERTY_CATEGORY"] != "Exempt")
    & (gdf["TAXCODE"].isin(["T", "P"]))
)
gdf.loc[vacant_override_mask, "PROPERTY_CATEGORY"] = "Vacant Land"

print("Property category distribution:")
display(gdf["PROPERTY_CATEGORY"].value_counts(dropna=False))

In [ ]:
# Mark fully exempt parcels
# TAXCODE='E' = fully exempt, TAXCODE='G' (government, if present) also excluded
gdf["full_exmp"] = (
    (gdf["TAXCODE"] == "E")
    | (gdf["LOCALTOTAL"] <= 0)
).astype(int)

print(f"Total parcels:            {len(gdf):,}")
print(f"Fully exempt:             {gdf['full_exmp'].sum():,}")
print(f"Taxable (city modeling):  {(gdf['full_exmp'] == 0).sum():,}")

# Validate: LOCALLAND + LOCALBUILDING ≈ LOCALTOTAL
gdf["local_sum"] = gdf["LOCALLAND"] + gdf["LOCALBUILDING"]
discrepancy = (gdf["local_sum"] - gdf["LOCALTOTAL"]).abs()
print(f"\nLOCALLAND + LOCALBUILDING vs LOCALTOTAL:")
print(f"  Max discrepancy: ${discrepancy.max():,.0f}")
print(f"  Mean discrepancy: ${discrepancy.mean():,.2f}")
print(f"  Parcels with >$100 discrepancy: {(discrepancy > 100).sum():,}")

## Step 4: Reconstruct Current Pittsburgh City Tax

This notebook models **the Pittsburgh city levy only** (8.06 mills in 2025), consistent with the approach in `st_paul_v2.ipynb` which isolates the city portion of the tax bill. The city levy is ~32.6% of the combined 24.74-mill bill.

**Order of operations** (Pittsburgh city levy):
1. `LOCALLAND` + `LOCALBUILDING` = `LOCALTOTAL` (homestead reduction already applied to LOCALBUILDING)
2. Apply 8.06 mills to `LOCALTOTAL`: `city_tax = LOCALTOTAL × 8.06 / 1000`
3. No post-tax credits modeled (Act 77 senior relief is administered separately and not in the parcel file)

**Validation**: The modeled city revenue should be consistent with Pittsburgh's annual budget. Pittsburgh's city general fund property tax revenue is approximately $130–150M/year from roughly $15–18B in assessed value at 8.06 mills.

In [ ]:
gdf["city_millage_rate"] = CITY_MILLAGE

current_city_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col="LOCALTOTAL",
    millage_rate_col="city_millage_rate",
    land_value_col="LOCALLAND",
    improvement_value_col="LOCALBUILDING",
    exemption_flag_col="full_exmp",
)

# Identity check
identity_rev = gdf.loc[gdf["full_exmp"] == 0, "LOCALTOTAL"].sum() * CITY_MILLAGE / 1000
print(f"Modeled city revenue:          ${current_city_revenue:>15,.0f}")
print(f"Identity check (LOCALTOTAL):   ${identity_rev:>15,.0f}")
print(f"Difference:                    {(current_city_revenue / identity_rev - 1) * 100:.4f}%")

# Full combined bill for reference
gdf["county_tax"] = gdf["COUNTYTOTAL"].clip(lower=0) * COUNTY_MILLAGE / 1000 * (1 - gdf["full_exmp"])
gdf["school_tax"] = gdf["LOCALTOTAL"].clip(lower=0) * SCHOOL_MILLAGE / 1000 * (1 - gdf["full_exmp"])
gdf["full_bill"] = gdf["current_tax"] + gdf["county_tax"] + gdf["school_tax"]

print(f"\nFor reference — full combined bill (all taxing bodies):")
print(f"  County levy revenue:    ${gdf['county_tax'].sum():>15,.0f}")
print(f"  City levy revenue:      ${gdf['current_tax'].sum():>15,.0f}")
print(f"  School levy revenue:    ${gdf['school_tax'].sum():>15,.0f}")
print(f"  Total combined:         ${gdf['full_bill'].sum():>15,.0f}")

In [ ]:
# Remove fully exempt parcels from the modeling set
df = gdf[gdf["full_exmp"] == 0].copy()
print(f"Removed {len(gdf) - len(df):,} fully exempt parcels")
print(f"Modeling parcels: {len(df):,}")
print(f"\nModeling parcel category breakdown:")
display(df["PROPERTY_CATEGORY"].value_counts())

## Step 5: Model LVT Scenarios

All scenarios are revenue-neutral: they raise the same total city levy revenue ($current_city_revenue) while shifting the millage split between land and improvements.

The land/improvement split uses the Allegheny County assessed values:
- **Land base**: `LOCALLAND` (unchanged by homestead — the homestead reduction comes off buildings, not land)
- **Improvement base**: `LOCALBUILDING` (already reduced by $15,000 homestead exclusion for eligible parcels)

Scenarios modeled:
1. **2:1 split-rate**: Land taxed at 2× the improvement rate (comparable to Pittsburgh's historical graded tax after 1979 when the ratio was ~5.77:1, down from an earlier era)
2. **4:1 split-rate**: More aggressive LVT shift
3. **10:1 split-rate**: Highly aggressive LVT (land bears 10× the improvement burden)
4. **Full building exemption**: Benchmark — 100% of revenue from land only

In [ ]:
def run_split_rate_scenario(df_input, ratio, revenue):
    """Run a split-rate LVT scenario and return results dict."""
    land_millage, improvement_millage, new_revenue, modeled = model_split_rate_tax(
        df=df_input,
        land_value_col="LOCALLAND",
        improvement_value_col="LOCALBUILDING",
        current_revenue=revenue,
        land_improvement_ratio=ratio,
        exemption_flag_col="full_exmp",
    )
    sf_mask = modeled["PROPERTY_CATEGORY"] == "Single Family Residential"
    return {
        "scenario": f"Split-rate {int(ratio)}:1",
        "land_millage": land_millage,
        "improvement_millage": improvement_millage,
        "new_revenue": new_revenue,
        "median_sf_pct_change": modeled.loc[sf_mask, "tax_change_pct"].median(),
        "mean_city_pct_change": modeled["tax_change_pct"].mean(),
        "df": modeled,
    }


scenarios = [
    run_split_rate_scenario(df, 2.0, current_city_revenue),
    run_split_rate_scenario(df, 4.0, current_city_revenue),
    run_split_rate_scenario(df, 10.0, current_city_revenue),
]

print("\nScenario Summary:")
print(f"{'Scenario':<25} {'Land Mill':>10} {'Imp Mill':>10} {'Revenue':>15} {'Med SF %Chg':>12}")
print("-" * 75)
for s in scenarios:
    print(
        f"{s['scenario']:<25} {s['land_millage']:>10.4f} {s['improvement_millage']:>10.4f}"
        f" ${s['new_revenue']:>13,.0f} {s['median_sf_pct_change']:>11.1f}%"
    )

## Step 6: Property Category Impact — 4:1 Scenario

The 4:1 scenario is used as the primary exhibit, consistent with the St. Paul notebooks. Allegheny County's aggregate improvement ratio (building share of total local value) determines which property types gain or lose under LVT.

In [ ]:
pittsburgh_4to1 = next(s for s in scenarios if "4:1" in s["scenario"])["df"].copy()

category_summary = calculate_category_tax_summary(
    df=pittsburgh_4to1,
    category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    pct_threshold=10.0,
)

print_category_tax_summary(
    summary_df=category_summary,
    title="4:1 Split-Rate City Tax Impact by Property Category — Pittsburgh, PA",
    pct_threshold=10.0,
)

display(category_summary)

In [ ]:
# Horizontal property-category impact bar chart
filtered = category_summary[
    (category_summary["median_tax_change_pct"] != 0)
    & (category_summary["property_count"] > 0)
].copy()

categories = filtered["PROPERTY_CATEGORY"].tolist()
counts = filtered["property_count"].tolist()
median_pct_change = filtered["median_tax_change_pct"].tolist()
median_dollar_change = filtered["median_tax_change"].tolist()
total_tax_change = filtered["total_tax_change_dollars"].tolist()

sorted_idx = np.argsort(median_pct_change)
categories = [categories[i] for i in sorted_idx]
counts = [counts[i] for i in sorted_idx]
median_pct_change = [median_pct_change[i] for i in sorted_idx]
median_dollar_change = [median_dollar_change[i] for i in sorted_idx]
total_tax_change = [total_tax_change[i] for i in sorted_idx]

bar_colors = ["#8B0000" if val > 0 else "#228B22" for val in median_pct_change]

bar_height = 0.75
fig_height = max(len(categories) * 0.8 + 1.2, 4)
fig, ax = plt.subplots(figsize=(17, fig_height))
y = np.arange(len(categories))

ax.barh(y, median_pct_change, color=bar_colors, edgecolor="none", height=bar_height, alpha=0.92, zorder=2)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.axvline(0, color="black", linewidth=0.8, zorder=3)

max_abs = max(abs(v) for v in median_pct_change) if median_pct_change else 1
label_offset = max_abs * 0.03

for i, (cat, count, pct, dollar, total) in enumerate(
    zip(categories, counts, median_pct_change, median_dollar_change, total_tax_change)
):
    # Category label on left
    ax.text(-max_abs * 1.02, i, cat, ha="right", va="center", fontsize=9.5, color="#333333")
    # Parcel count
    ax.text(-max_abs * 1.02, i - 0.28, f"n={count:,}", ha="right", va="center", fontsize=7.5, color="#888888")
    # Percentage label inside/outside bar
    x_label = pct + label_offset if pct >= 0 else pct - label_offset
    ha = "left" if pct >= 0 else "right"
    sign = "+" if pct > 0 else ""
    ax.text(x_label, i, f"{sign}{pct:.1f}%", ha=ha, va="center", fontsize=9, fontweight="bold")
    # Dollar change on right
    sign_d = "+" if dollar > 0 else ""
    ax.text(max_abs * 1.02, i, f"{sign_d}${dollar:,.0f}/yr", ha="left", va="center", fontsize=8.5, color="#555555")

ax.set_xlim(-max_abs * 1.7, max_abs * 1.7)
ax.set_ylim(-0.6, len(categories) - 0.4)
ax.set_title(
    "4:1 Split-Rate Land Value Tax — Median City Tax Change by Property Type\nPittsburgh, PA (City Levy Only, 2025)",
    fontsize=13, fontweight="bold", pad=12,
)
plt.tight_layout()
plt.savefig(data_dir / "pittsburgh_4to1_category_impact.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7: Multi-Scenario Comparison

Compare across scenarios to see how the land/improvement ratio affects key property types.

In [ ]:
# Compare median tax change % for key categories across all scenarios
key_categories = [
    "Single Family Residential",
    "Small Multi-Family (2-4 units)",
    "Large Multi-Family (5+ units)",
    "Commercial",
    "Vacant Land",
]

rows = []
for s in scenarios:
    summary = calculate_category_tax_summary(
        df=s["df"],
        category_col="PROPERTY_CATEGORY",
        current_tax_col="current_tax",
        new_tax_col="new_tax",
    )
    for cat in key_categories:
        row = summary[summary["PROPERTY_CATEGORY"] == cat]
        if len(row) > 0:
            rows.append({
                "Scenario": s["scenario"],
                "Category": cat,
                "Median % Change": row["median_tax_change_pct"].values[0],
                "Count": row["property_count"].values[0],
                "Land Millage": s["land_millage"],
                "Imp Millage": s["improvement_millage"],
            })

comparison_df = pd.DataFrame(rows)
pivot = comparison_df.pivot(index="Category", columns="Scenario", values="Median % Change")
print("Median city tax change (%) by scenario and property category:")
display(pivot.round(1))

## Step 8: Vacant Land and Parking Analysis

Pittsburgh has significant amounts of vacant land and surface parking, particularly in the Hill District, Hazelwood, and former industrial corridors. Under LVT, these parcels would face substantially higher taxes since they hold land without improvements.

In [ ]:
vacant_results = analyze_vacant_land(
    df=pittsburgh_4to1,
    land_value_col="LOCALLAND",
    improvement_value_col="LOCALBUILDING",
    property_type_col="PROPERTY_CATEGORY",
    vacant_identifier="Vacant Land",
    neighborhood_col="MUNIDESC",
    owner_col="OWNERCODE" if "OWNERCODE" in pittsburgh_4to1.columns else None,
    exemption_flag_col="full_exmp",
)
print_vacant_land_summary(vacant_results)

parking_results = analyze_parking_lots(
    df=pittsburgh_4to1,
    land_value_col="LOCALLAND",
    improvement_value_col="LOCALBUILDING",
    property_type_col="PROPERTY_CATEGORY",
    parking_identifier="Transportation - Parking",
    exemption_flag_col="full_exmp",
)
print_parking_analysis_summary(parking_results)

## Step 9: Census Demographics and Equity Analysis

Spatial join of parcels to Census block groups using:
- **County FIPS**: `42003` (Allegheny County, Pennsylvania)
- **ACS year**: 2022
- **Join method**: parcel centroids to block group polygons in EPSG:3857

Equity quintile analysis shows whether the 4:1 LVT shift is progressive (lower-income neighborhoods benefit more) or regressive.

In [ ]:
census_api_key = os.getenv("CENSUS_API_KEY")
print("CENSUS_API_KEY loaded:", bool(census_api_key))

census_data, census_boundaries = get_census_data_with_boundaries(
    fips_code=ALLEGHENY_FIPS,
    year=2022,
    api_key=census_api_key,
)

if census_boundaries.crs is None:
    census_boundaries = census_boundaries.set_crs(epsg=4326)

if pittsburgh_4to1.crs != census_boundaries.crs:
    pittsburgh_4to1 = pittsburgh_4to1.to_crs(census_boundaries.crs)

df_geo = match_to_census_blockgroups(
    gdf=pittsburgh_4to1,
    census_gdf=census_boundaries,
)

print(f"Parcels with census block group: {len(df_geo):,}")
print(f"Census block groups available:   {len(census_boundaries):,}")
print("Key demographic columns:")
display([c for c in ["std_geoid", "median_income", "minority_pct", "black_pct", "total_pop"] if c in df_geo.columns])

In [ ]:
bg_summary = calculate_block_group_summary(
    df=df_geo,
    group_col="std_geoid",
    tax_change_col="tax_change",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
)
display(bg_summary.head(10))

In [ ]:
def create_quintile_summary(df_input, value_col, labels=None):
    if labels is None:
        labels = ["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"]
    work = df_input[df_input[value_col].notna()].copy()
    if value_col == "median_income":
        work = work[work[value_col] > 0].copy()
    work["quintile"] = pd.qcut(work[value_col], 5, labels=labels, duplicates="drop")
    summary = work.groupby("quintile", observed=True).agg(
        count=("tax_change", "count"),
        mean_tax_change=("tax_change", "mean"),
        median_tax_change=("tax_change", "median"),
        mean_tax_change_pct=("tax_change_pct", "mean"),
        median_tax_change_pct=("tax_change_pct", "median"),
        mean_value=(value_col, "mean"),
    ).reset_index()
    return summary


income_quintiles = create_quintile_summary(bg_summary, "median_income")
minority_quintiles = create_quintile_summary(bg_summary, "minority_pct")

print("Tax change by neighborhood income quintile:")
display(income_quintiles[["quintile", "count", "mean_value", "mean_tax_change", "median_tax_change_pct"]])

print("\nTax change by neighborhood minority share quintile:")
display(minority_quintiles[["quintile", "count", "mean_value", "mean_tax_change", "median_tax_change_pct"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

def plot_quintile_bars(ax, quintile_df, title, ylabel="Median City Tax Change (%)"):
    colors = ["#228B22" if v <= 0 else "#8B0000" for v in quintile_df["median_tax_change_pct"]]
    bars = ax.bar(
        quintile_df["quintile"].astype(str),
        quintile_df["median_tax_change_pct"],
        color=colors,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.5,
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel("Block Group Quintile", fontsize=9)
    for bar, val in zip(bars, quintile_df["median_tax_change_pct"]):
        sign = "+" if val > 0 else ""
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + (0.3 if val >= 0 else -0.5),
            f"{sign}{val:.1f}%",
            ha="center", va="bottom" if val >= 0 else "top",
            fontsize=8.5, fontweight="bold",
        )
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


plot_quintile_bars(
    axes[0], income_quintiles,
    "4:1 LVT — Tax Change by Neighborhood Income Quintile\n(Q1=Lowest Income)",
)
plot_quintile_bars(
    axes[1], minority_quintiles,
    "4:1 LVT — Tax Change by Neighborhood Minority Share Quintile\n(Q1=Least Minority)",
)

fig.suptitle(
    "Pittsburgh, PA — 4:1 Split-Rate LVT Equity Analysis (City Levy Only, 2025)",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(data_dir / "pittsburgh_4to1_equity.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 10: Summary and Policy Implications

Pittsburgh has unique historical context for LVT. The graded tax operated from ~1913 to 2001 and is widely credited with preventing land speculation during Pittsburgh's transition from a steel-based economy. Henry George's ideas were particularly influential in Pittsburgh and Pennsylvania more broadly.

Key takeaways from this model:
- Under the 4:1 scenario, **single-family homeowners** (who tend to have high improvement ratios) see modest tax reductions while **vacant land holders** face significantly higher taxes.
- **Commercial properties** with large land footprints (surface parking, underutilized downtown parcels) see the largest tax increases.
- The equity analysis reveals whether LVT is progressive in Pittsburgh's context — given the city's significant racial and economic segregation across neighborhoods.

The model currently covers the **city levy only** (8.06 mills). A full-bill analysis would require coordinating with Allegheny County (6.43 mills) and the Pittsburgh School District (10.25 mills), which may require separate legislative action.

In [ ]:
# Final model summary
s4 = next(s for s in scenarios if "4:1" in s["scenario"])
s2 = next(s for s in scenarios if "2:1" in s["scenario"])

print("=" * 70)
print("PITTSBURGH LVT MODEL SUMMARY — City Levy Only (2025)")
print("=" * 70)
print(f"Current city millage:          {CITY_MILLAGE} mills")
print(f"Current city revenue:          ${current_city_revenue:,.0f}")
print(f"Taxable parcels modeled:       {len(df):,}")
print()
print("2:1 Split-Rate Scenario:")
print(f"  Land millage:              {s2['land_millage']:.4f} mills")
print(f"  Improvement millage:       {s2['improvement_millage']:.4f} mills")
print(f"  Median SF residential:     {s2['median_sf_pct_change']:+.1f}% city tax change")
print()
print("4:1 Split-Rate Scenario:")
print(f"  Land millage:              {s4['land_millage']:.4f} mills")
print(f"  Improvement millage:       {s4['improvement_millage']:.4f} mills")
print(f"  Median SF residential:     {s4['median_sf_pct_change']:+.1f}% city tax change")
print()
print("Data sources:")
print("  Parcel data:  WPRDC Allegheny County Master File (2024)")
print("  Geometry:     Allegheny County GIS OPENDATA Parcels MapServer")
print("  Millage:      Allegheny County Tax Collectors 2025")
print("  CLR (2025):   52.7% (PA STEB)")